# 02. Limpieza y validación de datos — Tikal y Flores (Petén)

**Objetivo:** auditar los CSV generados por `src.etl`, documentar inconsistencias y verificar reglas metodológicas. Este notebook **no altera** los archivos originales ni los procesados.

## 1. Configuración
Ejecutar desde VS Code con el kernel del entorno `.venv`. Las rutas se calculan desde la carpeta del proyecto.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src" / "validaciones.py").exists(), "No se encuentra src/validaciones.py"
sys.path.insert(0, str(ROOT))
from src.validaciones import evaluar, FILES
DATA = ROOT / "data" / "processed"
print("Proyecto:", ROOT)
print("Datos:", DATA)

Proyecto: c:\Users\Emadlg\Desktop\UVG\proyecto3_Data
Datos: c:\Users\Emadlg\Desktop\UVG\proyecto3_Data\data\processed


## 2. Auditoría automática
Se comprueban existencia, columnas, duplicados exactos, fechas, valores numéricos, rangos y reglas particulares.

In [2]:
reporte = evaluar(DATA)
print("Errores:", len(reporte["errores"]))
print("Advertencias:", len(reporte["advertencias"]))
print("Archivos leídos:", len(reporte["archivos"]), "/", len(FILES))
display(pd.DataFrame([{"archivo":k, "filas":v["filas"], "duplicados":v["duplicados_exactos"], "fecha_min":v.get("fecha_min"), "fecha_max":v.get("fecha_max")} for k,v in reporte["archivos"].items()]))

Errores: 1
Advertencias: 2
Archivos leídos: 9 / 9


,archivo,filas,duplicados,fecha_min,fecha_max
0,movilidad_mensual_peten.csv,60,0,2025-09-01,2026-08-01
1,movilidad_diaria_peten.csv,972,0,2025-04-11,2026-09-15
2,tikal_mensual.csv,116,0,2017-01-01,2026-08-01
3,aimm_pasajeros_mensual.csv,248,0,2022-01-01,2026-04-01
4,aimm_operaciones_mensual.csv,104,0,2022-01-01,2026-04-01
5,ocupacion_peten.csv,36,0,2023-01-01,2025-12-01
6,oferta_hotelera_peten.csv,15,0,None,None
7,migracion_nacional_mensual.csv,210,0,2009-01-01,2026-06-01
8,comparacion_fuentes_peten.csv,6,0,2025-09-01,2026-04-01


## 3. Hallazgos y advertencias
Los avisos requieren interpretación, no necesariamente corrección automática. Nunca transformar un dato faltante en cero.

In [3]:
for mensaje in reporte["errores"]: print("ERROR:", mensaje)
for mensaje in reporte["advertencias"]: print("AVISO:", mensaje)
if not reporte["errores"]: print("Sin errores críticos según las reglas implementadas.")

ERROR: movilidad_mensual_peten.csv: estadia_media_dias contiene valores vacíos/no numéricos
AVISO: Tikal: 7 meses con cero; confirmar cierre o ausencia de datos
AVISO: Oferta hotelera: interpretar cortes como inventario, no sumar trimestres ni anualizar 2026 parcial


## 4. Cobertura mensual por fuente
La base diaria no se debe sumar para obtener visitantes mensuales; aquí solo se cuentan fechas disponibles.

In [4]:
filas=[]
for nombre, info in reporte["archivos"].items():
    if "fecha" not in info["columnas"]: continue
    d=pd.read_csv(DATA/nombre, usecols=["fecha"])
    meses=pd.to_datetime(d["fecha"], errors="coerce").dt.to_period("M")
    filas.append({"fuente":nombre,"meses_distintos":meses.nunique(),"inicio":str(meses.min()),"fin":str(meses.max())})
display(pd.DataFrame(filas))

,fuente,meses_distintos,inicio,fin
0,movilidad_mensual_peten.csv,10,2025-09,2026-08
1,movilidad_diaria_peten.csv,13,2025-04,2026-09
2,tikal_mensual.csv,116,2017-01,2026-08
3,aimm_pasajeros_mensual.csv,52,2022-01,2026-04
4,aimm_operaciones_mensual.csv,52,2022-01,2026-04
5,ocupacion_peten.csv,36,2023-01,2025-12
6,migracion_nacional_mensual.csv,210,2009-01,2026-06
7,comparacion_fuentes_peten.csv,6,2025-09,2026-04


## 5. Movilidad: reglas de estadía y subdestinos
La estadía media ponderada se calcula únicamente en el segmento **mayor a 1 día**. Los subdestinos no se suman para obtener visitantes únicos de Petén.

In [5]:
mov = pd.read_csv(DATA/"movilidad_mensual_peten.csv")
noct = mov[mov["segmento"].str.lower().eq("mayor a 1 dia")].copy()
noct["estadia_recalculada"] = noct["dias_persona"] / noct["visitantes"].replace(0,float("nan"))
noct["diferencia"] = (noct["estadia_recalculada"]-noct["estadia_media_dias"]).abs()
display(noct[["fecha","subdestino","visitantes","dias_persona","estadia_media_dias","diferencia"]].head(12))
print("Diferencia máxima:",noct["diferencia"].max())

,fecha,subdestino,visitantes,dias_persona,estadia_media_dias,diferencia
0,2025-09-01,El Remate Y Alrededores Lago,28469.221902,143502.261789,5.040611,0.000000e+00
2,2025-09-01,Isla De Flores,34594.799822,209026.648247,6.042141,0.000000e+00
4,2025-09-01,Mundo Maya Aeropuerto,36358.438642,187611.832392,5.160063,8.881784e-16
6,2025-10-01,El Remate Y Alrededores Lago,30808.712989,158122.194333,5.132386,0.000000e+00
8,2025-10-01,Isla De Flores,36040.106153,214584.971609,5.954060,1.776357e-15
10,2025-10-01,Mundo Maya Aeropuerto,38792.339133,203917.654676,5.256648,0.000000e+00
12,2025-11-01,El Remate Y Alrededores Lago,33440.431701,158877.204542,4.751051,8.881784e-16
14,2025-11-01,Isla De Flores,37510.399442,203681.369787,5.429997,8.881784e-16
16,2025-11-01,Mundo Maya Aeropuerto,43639.408463,212404.675765,4.867268,0.000000e+00
18,2025-12-01,El Remate Y Alrededores Lago,39599.957003,172100.628912,4.345980,0.000000e+00


Diferencia máxima: 1.7763568394002505e-15


## 6. Tikal: meses con cero
Se listan sin reemplazarlos por valores inventados. Su significado debe confirmarse con la documentación de la fuente.

In [6]:
tikal=pd.read_csv(DATA/"tikal_mensual.csv")
display(tikal.loc[tikal["total"].eq(0), ["fecha","nacionales","extranjeros","otros","total"]])

,fecha,nacionales,extranjeros,otros,total
0,2017-01-01,0,0,0,0
39,2020-04-01,0,0,0,0
40,2020-05-01,0,0,0,0
41,2020-06-01,0,0,0,0
42,2020-07-01,0,0,0,0
43,2020-08-01,0,0,0,0
44,2020-09-01,0,0,0,0


## 7. Aeropuerto y triangulación
Pasajeros, operaciones, boletos y telefonía representan universos distintos. El cruce se realiza por mes y no supone equivalencia entre unidades.

In [7]:
cmp = pd.read_csv(DATA/"comparacion_fuentes_peten.csv")
display(cmp)
print("Meses disponibles para comparación:",len(cmp))

,fecha,boletos_tikal,pasajeros_aimm,movilidad_poligono_aeropuerto
0,2025-09-01,17758,27377,99370.640787
1,2025-10-01,23408,31931,104281.761180
2,2025-11-01,41411,38603,112807.669063
3,2025-12-01,53209,37335,137352.271183
4,2026-03-01,45946,38450,127506.123433
5,2026-04-01,44389,37673,130191.176600


Meses disponibles para comparación: 6


## 8. Ocupación y oferta hotelera
Verificar la escala de ocupación (proporción o porcentaje). La oferta hotelera representa inventarios en cortes temporales, no flujos acumulables.

In [8]:
ocup=pd.read_csv(DATA/"ocupacion_peten.csv")
oferta=pd.read_csv(DATA/"oferta_hotelera_peten.csv")
display(ocup.head())
display(oferta.head())
print("Rango ocupación:",ocup["indice_ocupacion"].min(),ocup["indice_ocupacion"].max())

,fecha,indice_ocupacion
0,2023-01-01,0.64
1,2023-02-01,0.69
2,2023-03-01,0.63
3,2023-04-01,0.72
4,2023-05-01,0.86


,anio,trimestre,tipo_alojamiento,establecimientos,habitaciones,plazas_cama
0,2022,4,Hospedaje,157,2345,6325
1,2022,4,Hotel,83,1540,4668
2,2022,4,Pensión,16,219,684
3,2023,4,Hospedaje,145,2180,5987
4,2023,4,Hotel,75,1488,4518


Rango ocupación: 0.33 0.96


## 9. Conclusiones metodológicas
- Mantener la distinción entre valores observados, estimaciones y ausencia de cobertura.
- Investigar los ceros de Tikal antes de interpretar tendencias.
- No sumar subdestinos ni presencias diarias para estimar visitantes únicos.
- Documentar la cobertura de cada fuente y utilizar únicamente ventanas comparables.
- Ejecutar `python -m src.validaciones` y `python -m pytest tests/test_validaciones.py -v` desde la raíz.